# Scripts SQLite para PokéAPI

Este notebook contiene el material completo del proyecto de Pokédex local con SQLite y la PokéAPI:

1. **Script 1** — Inicialización de la base de datos (tablas, índices, vistas, triggers)
2. **Script 2** — Descarga masiva de todos los Pokémon desde la PokéAPI
3. **Script 3** — Consultas interactivas offline con fallback a la PokéAPI
4. **Cuestionario** — 10 preguntas de repaso
5. **Ejercicios prácticos** — 10 ejercicios para aplicar lo aprendido

## Script 1: Inicialización de la base de datos de la Pokédex local

Este script crea la estructura necesaria en `pokemon.db` para almacenar la información descargada desde la PokéAPI. Está pensado para ejecutarse **una sola vez** antes de la descarga masiva.

### ¿Qué hace?

1. Se conecta a `pokemon.db` (lo crea si no existe).
2. Activa `PRAGMA foreign_keys = ON` para respetar las relaciones entre tablas.
3. Crea las tablas: `tipos`, `pokemon`, `pokemon_tipo` (relación muchos a muchos) y `log_nuevos` (auditoría).
4. Crea índices sobre `pokemon.nombre` y `pokemon_tipo.tipo_id`.
5. Crea la vista `vista_pokemon_tipos` con tipos concatenados.
6. Crea el trigger `trg_nuevo_pokemon` para auditoría automática.

Todas las sentencias usan `IF NOT EXISTS` o `DROP TRIGGER IF EXISTS` para ser reejecutable sin errores.

In [1]:
import sqlite3

# ------------------------------------------------------------
# 1. Conexión a la base de datos (se crea si no existe)
# ------------------------------------------------------------
# Si el archivo pokemon.db no existe, sqlite3 lo crea automáticamente.
# Usamos el administrador de contexto 'with' para asegurar que la conexión
# se cierre correctamente al finalizar.
def inicializar_db(ruta="pokemon.db"):
    with sqlite3.connect(ruta) as conexion:
        conexion.execute("PRAGMA foreign_keys = ON")
        cursor = conexion.cursor()

        _crear_tablas(cursor)
        _crear_indices(cursor)
        _crear_vista(cursor)
        _crear_trigger(cursor)

        # Llamada explícita a commit: el bloque 'with' solo cierra la conexión,
        # no hace commit automáticamente.
        conexion.commit()

    print(f"Base de datos '{ruta}' inicializada correctamente.")


# ------------------------------------------------------------
# 2. Creación de tablas
# ------------------------------------------------------------
# Usamos CREATE TABLE IF NOT EXISTS para que el script sea idempotente.
def _crear_tablas(cursor):
    # Tabla de tipos de Pokémon
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS tipos (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nombre TEXT UNIQUE NOT NULL
        )
    """)

    # Tabla principal de Pokémon
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS pokemon (
            id INTEGER PRIMARY KEY,
            nombre TEXT NOT NULL,
            altura REAL,
            peso REAL,
            experiencia_base INTEGER
        )
    """)

    # Tabla intermedia para la relación muchos a muchos
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS pokemon_tipo (
            pokemon_id INTEGER NOT NULL,
            tipo_id INTEGER NOT NULL,
            PRIMARY KEY (pokemon_id, tipo_id),
            FOREIGN KEY (pokemon_id) REFERENCES pokemon(id) ON DELETE CASCADE,
            FOREIGN KEY (tipo_id) REFERENCES tipos(id) ON DELETE CASCADE
        )
    """)

    # Tabla de auditoría para registrar cada nuevo Pokémon insertado
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS log_nuevos (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            pokemon_id INTEGER NOT NULL,
            fecha TEXT DEFAULT (datetime('now')),
            FOREIGN KEY (pokemon_id) REFERENCES pokemon(id) ON DELETE CASCADE
        )
    """)


# ------------------------------------------------------------
# 3. Creación de índices
# ------------------------------------------------------------
# Mejoran el rendimiento de las consultas por nombre y por tipo.
def _crear_indices(cursor):
    # Índice para búsquedas por nombre de Pokémon
    cursor.execute("""
        CREATE INDEX IF NOT EXISTS idx_pokemon_nombre ON pokemon (nombre)
    """)

    # Índice para obtener todos los Pokémon de un tipo concreto
    cursor.execute("""
        CREATE INDEX IF NOT EXISTS idx_pokemon_tipo_tipo ON pokemon_tipo (tipo_id)
    """)


# ------------------------------------------------------------
# 4. Creación de la vista
# ------------------------------------------------------------
# "vista_pokemon_tipos" devuelve cada Pokémon con sus tipos
# concatenados (separados por coma) en una sola columna.
def _crear_vista(cursor):
    cursor.execute("""
        CREATE VIEW IF NOT EXISTS vista_pokemon_tipos AS
        SELECT p.id, p.nombre,
               GROUP_CONCAT(t.nombre, ', ') AS tipos
        FROM pokemon p
        JOIN pokemon_tipo pt ON p.id = pt.pokemon_id
        JOIN tipos t ON pt.tipo_id = t.id
        GROUP BY p.id
    """)


# ------------------------------------------------------------
# 5. Creación del trigger
# ------------------------------------------------------------
# Para que el script sea reejecutable sin errores, primero
# eliminamos el trigger si ya existía y luego lo creamos.
def _crear_trigger(cursor):
    cursor.execute("DROP TRIGGER IF EXISTS trg_nuevo_pokemon")

    cursor.execute("""
        CREATE TRIGGER trg_nuevo_pokemon
        AFTER INSERT ON pokemon
        FOR EACH ROW
        BEGIN
            INSERT INTO log_nuevos (pokemon_id) VALUES (NEW.id);
        END
    """)

if __name__ == "__main__":
    inicializar_db()

Base de datos 'pokemon.db' inicializada correctamente.


### Cómo utilizar

1. Ejecuta la celda anterior directamente en el notebook.  
   También puedes guardarla como script (`inicializar_db.py`) y ejecutarla con:
   ```bash
   python inicializar_db.py
   ```
2. Se creará (o actualizará) `pokemon.db` en el directorio actual.
3. Ya estará lista para recibir datos con el Script 2.

### Notas importantes

- **Sin conexión a internet**: este script solo define estructuras, no requiere red.
- **Idempotente**: todas las sentencias usan `IF NOT EXISTS` o `DROP IF EXISTS`.
- **Integridad referencial**: `PRAGMA foreign_keys = ON` asegura las relaciones.
- **Auditoría automática**: `log_nuevos.fecha` se rellena automáticamente con `datetime('now')`.
- Las funciones auxiliares («privadas» con prefijo `_`) organizan el código y lo hacen reutilizable.

## Script 2: Descarga de TODOS los Pokémon desde la PokéAPI

Este script se conecta a la **PokéAPI pública**, obtiene **todos los Pokémon disponibles** (1000+) y los guarda en `pokemon.db`.

### ¿Qué hace?

1. Se conecta a la base de datos local.
2. Descubre el total de Pokémon preguntando a `/pokemon?limit=1`.
3. Define la función `guardar_pokemon` que inserta tipos, Pokémon y relaciones (idempotente).
4. Itera desde ID 1 hasta el total, descargando cada Pokémon.
5. Gestiona errores de red, datos mal formados e IDs inexistentes (404).
6. Incluye una pausa configurable entre peticiones.

El script es **idempotente**: si se interrumpe, puede reanudarse sin duplicar datos gracias a `INSERT OR IGNORE`.

In [ ]:
import sqlite3
import requests
import time

# Constantes con nombre (evitan números mágicos)
TIMEOUT_SEGUNDOS = 10
DELAY_ENTRE_PETICIONES = 0.3
TOTAL_FALLBACK = 2000
API_BASE = "https://pokeapi.co/api/v2"


# ------------------------------------------------------------
# 1. Función de inserción de un Pokémon
# ------------------------------------------------------------
def guardar_pokemon(conexion, datos_json):
    # Almacena un Pokémon y sus tipos en la base de datos.
    # Ignora inserciones duplicadas (idempotente).
    cursor = conexion.cursor()

    poke_id = datos_json["id"]
    nombre = datos_json["name"].lower()
    altura = datos_json["height"]
    peso = datos_json["weight"]
    experiencia = datos_json.get("base_experience")

    tipos_nombres = [t["type"]["name"] for t in datos_json["types"]]

    # Insertar tipos (si no existen)
    for tipo_nombre in tipos_nombres:
        cursor.execute(
            "INSERT OR IGNORE INTO tipos (nombre) VALUES (?)",
            (tipo_nombre,)
        )

    # Insertar Pokémon
    cursor.execute(
        "INSERT OR IGNORE INTO pokemon (id, nombre, altura, peso, experiencia_base) VALUES (?, ?, ?, ?, ?)",
        (poke_id, nombre, altura, peso, experiencia)
    )

    # Insertar relaciones en pokemon_tipo
    for tipo_nombre in tipos_nombres:
        cursor.execute("SELECT id FROM tipos WHERE nombre = ?", (tipo_nombre,))
        fila = cursor.fetchone()
        tipo_id = fila[0]  # Siempre existe por el INSERT OR IGNORE anterior
        cursor.execute(
            "INSERT OR IGNORE INTO pokemon_tipo (pokemon_id, tipo_id) VALUES (?, ?)",
            (poke_id, tipo_id)
        )


# ------------------------------------------------------------
# 2. Obtener el número total de Pokémon disponibles
# ------------------------------------------------------------
def obtener_total_pokemon():
    print("Consultando el número total de Pokémon en la API...")
    try:
        respuesta = requests.get(f"{API_BASE}/pokemon?limit=1", timeout=TIMEOUT_SEGUNDOS)
        respuesta.raise_for_status()
        total = respuesta.json()["count"]
        print(f"Se van a descargar {total} Pokémon.")
        return total
    except requests.exceptions.RequestException as e:
        print(f"No se pudo obtener el total de Pokémon: {e}")
        print(f"Se usará un número alto por defecto ({TOTAL_FALLBACK}).")
        return TOTAL_FALLBACK


# ------------------------------------------------------------
# 3. Descarga de todos los Pokémon
# ------------------------------------------------------------
def descargar_y_guardar(conexion, pokemon_id, total):
    url = f"{API_BASE}/pokemon/{pokemon_id}"
    try:
        respuesta = requests.get(url, timeout=TIMEOUT_SEGUNDOS)
        if respuesta.status_code == 404:
            # La API no tiene un Pokémon con ese ID (es normal que haya huecos)
            print(f"  [{pokemon_id}/{total}] No existe (404). Se omite.")
            return True  # No hay error, solo no existe

        respuesta.raise_for_status()
        datos = respuesta.json()
        guardar_pokemon(conexion, datos)
        print(f"  [{pokemon_id}/{total}] {datos['name']} guardado.")
        return True

    except requests.exceptions.RequestException as e:
        print(f"  [{pokemon_id}/{total}] Error de red: {e}")
        return False
    except Exception as e:
        print(f"  [{pokemon_id}/{total}] Error inesperado: {e}")
        return False


# ------------------------------------------------------------
# 4. Ejecución principal
# ------------------------------------------------------------
with sqlite3.connect("pokemon.db") as conexion:
    conexion.execute("PRAGMA foreign_keys = ON")

    total_pokemon = obtener_total_pokemon()

    for pokemon_id in range(1, total_pokemon + 1):
        descargar_y_guardar(conexion, pokemon_id, total_pokemon)
        time.sleep(DELAY_ENTRE_PETICIONES)

    conexion.commit()
    print("¡Descarga finalizada! Todos los Pokémon disponibles han sido guardados en 'pokemon.db'.")

### Notas sobre el funcionamiento

- **Total dinámico**: consulta `count` desde la API (actualmente ~1300). Siempre descarga todo lo disponible.
- **Huecos en la numeración**: IDs sin Pokémon asignado devuelven 404 y se omiten.
- **Tiempo de ejecución**: con ~1000+ Pokémon y 0.3s de pausa, puede durar varios minutos.
- **Idempotencia**: `INSERT OR IGNORE` permite reanudar descargas interrumpidas sin duplicar datos.
- **Dependencia extra**: necesita `requests`. Instala con `pip install requests`.

### Posibles ajustes

- Para limitar a una generación, cambia `range(1, total_pokemon + 1)` por el rango deseado.
- Ajusta `DELAY_ENTRE_PETICIONES` si encuentras errores de límite de tasa.
- Vuelve a ejecutar periódicamente para añadir Pokémon nuevos si la API ha crecido.
- Las constantes al inicio del script (`TIMEOUT_SEGUNDOS`, `API_BASE`, etc.) se pueden modificar sin tocar el resto del código.

## Script 3: Consultas offline con fallback a la PokéAPI

**Punto de entrada interactivo** de la Pokédex local. Realiza consultas **sin conexión** siempre que sea posible, y recurre a la PokéAPI como respaldo cuando no hay datos locales.

### ¿Qué ofrece?

- **Menú interactivo** con 6 opciones:
  1. Buscar Pokémon por nombre o ID
  2. Listar Pokémon de un tipo
  3. Top 10 Pokémon más pesados
  4. Altura media de todos los Pokémon
  5. Exportar Pokédex a CSV
  6. Salir
- Cada función comprueba si la conexión a la BD es válida; si no, deriva a la API o muestra un mensaje.
- Consultas parametrizadas y manejo de errores de red.

### Requisitos

- Python 3 + bibliotecas: `sqlite3`, `csv`, `os` (estándar) y `requests` (`pip install requests`).
- Las opciones 3, 4 y 5 requieren base de datos local poblada.
- Las opciones 1 y 2 funcionan incluso sin base de datos (vía API).

In [ ]:
import sqlite3
import requests
import os
import csv

# Constantes
API_BASE = "https://pokeapi.co/api/v2"
TIMEOUT_SEGUNDOS = 10
DB_FALLBACK_MENSAJE = "Opci\u00f3n no disponible sin base de datos local."


# ------------------------------------------------------------
# 1. Conexi\u00f3n a la base de datos local
# ------------------------------------------------------------
def conectar_db():
    # Intenta abrir pokemon.db. Retorna conexi\u00f3n o None.
    try:
        if not os.path.exists("pokemon.db"):
            print("Aviso: no se encuentra 'pokemon.db'. Se trabajar\u00e1 solo con la API.")
            return None
        conexion = sqlite3.connect("pokemon.db")
        conexion.execute("PRAGMA foreign_keys = ON")
        return conexion
    except (sqlite3.Error, OSError) as e:
        print(f"Error al conectar con la base de datos: {e}")
        return None


# ------------------------------------------------------------
# 2. Funciones auxiliares de consulta
# ------------------------------------------------------------
def _consulta_tipos_por_id(cursor, pokemon_id):
    # Obtiene los tipos concatenados de un Pok\u00e9mon por su ID.
    cursor.execute("""
        SELECT GROUP_CONCAT(t.nombre, ', ')
        FROM pokemon_tipo pt
        JOIN tipos t ON pt.tipo_id = t.id
        WHERE pt.pokemon_id = ?
    """, (pokemon_id,))
    return cursor.fetchone()[0] or "sin tipo"


def buscar_por_id(conexion, pokemon_id):
    # Busca un Pok\u00e9mon en la BD local por ID num\u00e9rico exacto.
    cursor = conexion.cursor()
    cursor.execute("""
        SELECT id, nombre, altura, peso, experiencia_base
        FROM pokemon WHERE id = ?
    """, (pokemon_id,))
    encontrado = cursor.fetchone()
    if not encontrado:
        return None

    tipos = _consulta_tipos_por_id(cursor, pokemon_id)
    return {
        "id": encontrado[0],
        "nombre": encontrado[1],
        "altura": encontrado[2],
        "peso": encontrado[3],
        "experiencia_base": encontrado[4],
        "tipos": tipos
    }


def buscar_por_nombre(conexion, nombre):
    # Busca un Pok\u00e9mon en la BD local por nombre exacto o parcial.
    cursor = conexion.cursor()
    cursor.execute("""
        SELECT id, nombre, altura, peso, experiencia_base
        FROM pokemon
        WHERE LOWER(nombre) = LOWER(?) OR LOWER(nombre) LIKE ?
    """, (nombre, f"%{nombre}%"))
    coincidencias = cursor.fetchall()
    if not coincidencias:
        return None

    # Si hay m\u00faltiples coincidencias, se muestra la primera.
    encontrado = coincidencias[0]
    tipos = _consulta_tipos_por_id(cursor, encontrado[0])
    return {
        "id": encontrado[0],
        "nombre": encontrado[1],
        "altura": encontrado[2],
        "peso": encontrado[3],
        "experiencia_base": encontrado[4],
        "tipos": tipos
    }


def buscar_local(conexion, nombre_o_id):
    # Busca un Pok\u00e9mon en local: por ID si es num\u00e9rico, por nombre si no.
    try:
        return buscar_por_id(conexion, int(nombre_o_id))
    except ValueError:
        return buscar_por_nombre(conexion, nombre_o_id)


def buscar_api(nombre_o_id):
    # Busca un Pok\u00e9mon en la Pok\u00e9API por nombre o ID.
    url = f"{API_BASE}/pokemon/{nombre_o_id.lower()}"
    try:
        respuesta = requests.get(url, timeout=TIMEOUT_SEGUNDOS)
        respuesta.raise_for_status()
        datos = respuesta.json()
        return {
            "id": datos["id"],
            "nombre": datos["name"].lower(),
            "altura": datos["height"],
            "peso": datos["weight"],
            "experiencia_base": datos.get("base_experience"),
            "tipos": ", ".join(t["type"]["name"] for t in datos["types"])
        }
    except requests.exceptions.RequestException as e:
        print(f"Error de conexi\u00f3n a la API: {e}")
    except Exception as e:
        print(f"Error al procesar la respuesta de la API: {e}")
    return None


# ------------------------------------------------------------
# 3. Funciones del men\u00fa
# ------------------------------------------------------------
def _mostrar_pokemon(pokemon, origen=""):
    # Formatea y muestra los datos de un Pok\u00e9mon en pantalla.
    if origen:
        print(origen)
    print(f"\n  ID:            {pokemon['id']}")
    print(f"  Nombre:        {pokemon['nombre'].capitalize()}")
    print(f"  Tipos:         {pokemon['tipos']}")
    print(f"  Altura:        {pokemon['altura']} dm ({pokemon['altura']/10} m)")
    print(f"  Peso:          {pokemon['peso']} hg ({pokemon['peso']/10} kg)")
    if pokemon['experiencia_base'] is not None:
        print(f"  Exp. base:     {pokemon['experiencia_base']}")
    else:
        print("  Exp. base:     --")


def opcion_buscar(conexion):
    nombre = input("Nombre o ID del Pok\u00e9mon: ").strip()
    if not nombre:
        print("Debe introducir un valor.")
        return

    pokemon = None
    if conexion:
        pokemon = buscar_local(conexion, nombre)
        if pokemon:
            _mostrar_pokemon(pokemon, "(Obtenido de la base de datos local)")
            return

    print("No encontrado en local. Consultando en la API...")
    pokemon = buscar_api(nombre)
    if pokemon:
        _mostrar_pokemon(pokemon)
    else:
        print("No se encontr\u00f3 el Pok\u00e9mon ni en local ni en la API.")


def _listar_pokemon_por_tipo_local(conexion, tipo):
    # Consulta los Pok\u00e9mon de un tipo desde la base de datos local.
    cursor = conexion.cursor()
    cursor.execute("""
        SELECT p.nombre
        FROM pokemon p
        JOIN pokemon_tipo pt ON p.id = pt.pokemon_id
        JOIN tipos t ON pt.tipo_id = t.id
        WHERE LOWER(t.nombre) = ?
        ORDER BY p.nombre
    """, (tipo,))
    return [row[0] for row in cursor.fetchall()]


def _listar_pokemon_por_tipo_api(tipo):
    # Consulta los Pok\u00e9mon de un tipo desde la Pok\u00e9API.
    url = f"{API_BASE}/type/{tipo}"
    try:
        respuesta = requests.get(url, timeout=TIMEOUT_SEGUNDOS)
        respuesta.raise_for_status()
        datos = respuesta.json()
        return [p["pokemon"]["name"] for p in datos["pokemon"]]
    except requests.exceptions.RequestException as e:
        print(f"Error al conectar con la API: {e}")
    except Exception as e:
        print(f"Error al procesar la respuesta: {e}")
    return None


def _imprimir_lista_pokemon(pokemon_list, tipo, origen):
    # Imprime una lista de nombres de Pok\u00e9mon formateada.
    print(f"\nPok\u00e9mon de tipo '{tipo}' {origen}:")
    for nombre in pokemon_list:
        print(f"  - {nombre.capitalize()}")
    print(f"Total: {len(pokemon_list)}")


def opcion_tipo(conexion):
    tipo = input("Introduce un tipo (ej. fire, water, electric): ").strip().lower()
    if not tipo:
        return

    if conexion:
        pokemon_list = _listar_pokemon_por_tipo_local(conexion, tipo)
        if pokemon_list:
            _imprimir_lista_pokemon(pokemon_list, tipo, "(desde BD local)")
            return

    # Fallback a API
    print("Tipo no encontrado en local. Consultando en la API...")
    pokemon_list = _listar_pokemon_por_tipo_api(tipo)
    if pokemon_list is None:
        return
    if pokemon_list:
        _imprimir_lista_pokemon(pokemon_list, tipo, "(desde API)")
    else:
        print("No se encontraron Pok\u00e9mon de ese tipo.")


def opcion_top10_pesados(conexion):
    if not conexion:
        print(DB_FALLBACK_MENSAJE)
        return
    cursor = conexion.cursor()
    cursor.execute("""
        SELECT nombre, peso
        FROM pokemon
        WHERE peso IS NOT NULL
        ORDER BY peso DESC
        LIMIT 10
    """)
    resultados = cursor.fetchall()
    print("\nTop 10 Pok\u00e9mon m\u00e1s pesados:")
    for nombre, peso in resultados:
        print(f"  {nombre.capitalize():<20} {peso} hg ({peso/10} kg)")


def opcion_altura_media(conexion):
    if not conexion:
        print(DB_FALLBACK_MENSAJE)
        return
    cursor = conexion.cursor()
    cursor.execute("SELECT AVG(altura) FROM pokemon WHERE altura IS NOT NULL")
    media = cursor.fetchone()[0]
    if media:
        print(f"\nAltura media de todos los Pok\u00e9mon: {media:.2f} dm ({media/10:.2f} m)")
    else:
        print("No se pudo calcular la altura media.")


def opcion_exportar_csv(conexion):
    if not conexion:
        print(DB_FALLBACK_MENSAJE)
        return
    archivo = input("Nombre del archivo CSV (p.ej. pokedex.csv): ").strip()
    if not archivo:
        archivo = "pokedex.csv"
    cursor = conexion.cursor()
    cursor.execute("SELECT * FROM vista_pokemon_tipos")
    with open(archivo, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["ID", "Nombre", "Tipos"])
        writer.writerows(cursor.fetchall())
    print(f"Pok\u00e9dex exportada a '{archivo}' correctamente.")


# ------------------------------------------------------------
# 4. Men\u00fa principal
# ------------------------------------------------------------
def _mostrar_menu():
    print("\n--- Men\u00fa ---")
    print("1. Buscar Pok\u00e9mon por nombre o ID")
    print("2. Listar Pok\u00e9mon de un tipo")
    print("3. Top 10 Pok\u00e9mon m\u00e1s pesados")
    print("4. Altura media de todos los Pok\u00e9mon")
    print("5. Exportar Pok\u00e9dex a CSV")
    print("6. Salir")


def _ejecutar_opcion(opcion, conexion):
    opciones = {
        "1": opcion_buscar,
        "2": opcion_tipo,
        "3": opcion_top10_pesados,
        "4": opcion_altura_media,
        "5": opcion_exportar_csv,
    }
    accion = opciones.get(opcion)
    if accion:
        accion(conexion)
        return True
    if opcion == "6":
        print("\u00a1Hasta luego!")
        return False
    print("Opci\u00f3n no v\u00e1lida.")
    return True


def main():
    print("=== Pok\u00e9dex Interactiva (Local + API) ===")
    conexion = conectar_db()

    while True:
        _mostrar_menu()
        opcion = input("Elige una opci\u00f3n: ").strip()
        if not _ejecutar_opcion(opcion, conexion):
            break

    if conexion:
        conexion.close()


if __name__ == "__main__":
    main()

### Cómo utilizar

1. Asegúrate de haber ejecutado los Scripts 1 y 2 al menos una vez.
2. Ejecuta la celda anterior e interactúa con el menú.
3. Si un Pokémon no está en local, se consultará automáticamente a la PokéAPI (requiere internet).
4. Las opciones 3, 4 y 5 solo funcionan con base de datos local.

### Comportamiento híbrido

- **Con BD local**: búsquedas instantáneas offline. Opciones 3 y 4 con SQL agregado.
- **Sin BD local**: solo opciones 1 y 2, usando la API en vivo.

### Notas importantes

- `buscar_local` acepta nombre (parcial) o ID numérico exacto.
- La exportación CSV usa la vista `vista_pokemon_tipos`.
- Para añadir más funcionalidades (ej. buscar por habilidad), sigue el mismo patrón de las funciones del menú.
- Las funciones con prefijo `_` son auxiliares internas que organizan el código y evitan repetición.

## Cuestionario (10 preguntas)

Responde brevemente cada pregunta.

1. **¿Por qué es necesario ejecutar `PRAGMA foreign_keys = ON` al abrir una conexión en SQLite? ¿Qué podría ocurrir si no se hace?**

2. **En la tabla `pokemon_tipo`, ¿por qué la clave primaria es compuesta `(pokemon_id, tipo_id)` en lugar de tener una sola columna autoincremental?**

3. **¿Qué ventaja tiene usar `INSERT OR IGNORE` en la tabla `tipos` durante la descarga masiva desde la API? ¿Qué pasaría si se usara `INSERT` simple?**

4. **Observa la función `guardar_pokemon` del Script 2. ¿En qué orden se realizan las inserciones y por qué se debe respetar ese orden?**

5. **¿Por qué se crea un índice sobre `tipo_id` en `pokemon_tipo` si la clave primaria ya incluye `pokemon_id` y `tipo_id`? ¿Qué consulta específica se beneficia de él?**

6. **La vista `vista_pokemon_tipos` utiliza `GROUP_CONCAT`. ¿Qué ocurriría con un Pokémon que tuviera dos tipos? ¿Cómo aparecería en la columna `tipos`?**

7. **¿Qué es una tabla intermedia y qué problema de modelado resuelve? Ilustra con el ejemplo de Pokémon y tipos.**

8. **En el Script 3, cuando no hay base de datos local, ¿cómo obtiene la lista de Pokémon de un tipo la función `opcion_tipo`? ¿Qué URL utiliza?**

9. **¿Por qué el trigger `trg_nuevo_pokemon` está definido como `AFTER INSERT` y no como `BEFORE INSERT`? ¿Qué información necesita del nuevo registro?**

10. **Explica la diferencia entre una vista y una tabla normal. ¿Los datos de la vista se almacenan físicamente en el archivo `.db`?**

## Ejercicios prácticos (10 ejercicios)

Realiza cada ejercicio modificando o creando pequeños scripts a partir de los ya proporcionados. Usa `pokemon.db` con datos descargados.

### Ejercicio 1 – Verificación del esquema

Crea un script Python que se conecte a `pokemon.db` y muestre por pantalla todas las tablas existentes, así como los índices y las vistas definidos.

*Pista: consulta la tabla `sqlite_master`.*

In [ ]:
# Ejercicio 1: Verificación del esquema
import sqlite3

conexion = sqlite3.connect("pokemon.db")
cursor = conexion.cursor()

# Mostrar tablas, índices y vistas desde sqlite_master

conexion.close()

### Ejercicio 2 – Ampliar la tabla `pokemon`

Añade una columna `color` (TEXT) a la tabla `pokemon`. Luego escribe un script que recorra los Pokémon locales (IDs del 1 al 10), obtenga su color desde la PokéAPI (`pokemon-species`) y actualice la base de datos.

> *No es necesario modificar el Script 2 completo, solo un pequeño programa adicional.*

### Ejercicio 3 – Búsqueda parcial mejorada

Modifica la función `buscar_por_nombre` del Script 3 para que, si se encuentran varios Pokémon que coinciden con el patrón, muestre todos ellos con un formato abreviado y permita al usuario elegir uno para ver los detalles completos.

### Ejercicio 4 – Auditoría con trigger

El trigger `trg_nuevo_pokemon` registra solo el `pokemon_id` y la fecha. Modifícalo para que también guarde el nombre del Pokémon. Si ya está poblada la tabla `log_nuevos`, puedes eliminarla y recrearla con la nueva estructura.

> *Escribe el nuevo código SQL del trigger y el script Python para aplicarlo.*

In [ ]:
-- Ejercicio 4: Nuevo trigger con nombre del Pokémon
-- (Ejecutar desde Python con sqlite3)

### Ejercicio 5 – Estadísticas por tipo

Escribe un programa Python que, sin usar la API, muestre cuántos Pokémon hay de cada tipo (ordenado de mayor a menor). Debe ejecutarse completamente offline.

> *Usa consultas con JOIN y GROUP BY.*

### Ejercicio 6 – Exportación selectiva

Amplía la opción 5 (exportar CSV) del Script 3 para que pregunte también por un tipo concreto y exporte únicamente los Pokémon de ese tipo, incluyendo las columnas ID, nombre, altura, peso y tipos concatenados.

### Ejercicio 7 – Respaldo online inteligente

En el Script 3, la función `buscar_api` se invoca si no hay datos locales. Modifica el programa para que, **si encuentra el Pokémon en la API, lo inserte automáticamente en la base de datos local** (solo si la base de datos existe), de manera que la próxima consulta ya sea offline.

### Ejercicio 8 – Validación de peso

Crea un trigger `BEFORE INSERT` en la tabla `pokemon` que impida insertar un Pokémon con peso negativo o nulo. Si se intenta, debe lanzar un error con `RAISE(ABORT, 'El peso debe ser positivo')`. Pruébalo intentando insertar manualmente un Pokémon con peso -1.

### Ejercicio 9 – Comparativa de rendimiento

Con la base de datos poblada con todos los Pokémon, escribe una consulta que muestre el plan de ejecución (`EXPLAIN QUERY PLAN`) al buscar un Pokémon por nombre. Luego borra el índice `idx_pokemon_nombre` (solo temporalmente, en una transacción que hagas rollback) y vuelve a ejecutar el plan. Observa la diferencia y redacta un comentario.

### Ejercicio 10 – Función definida por el usuario

Crea una función en Python que convierta el peso de hectogramos (hg) a kilogramos (kg) con un decimal y regístrala en SQLite con `create_function`. Después escribe una consulta SQL que use esa función para mostrar el nombre y el peso en kg de los 5 Pokémon más ligeros.

In [ ]:
# Ejercicio 10: Función definida por el usuario
import sqlite3

def peso_kg(hecto):
    return round(hecto / 10, 1)

conexion = sqlite3.connect("pokemon.db")
conexion.create_function("peso_kg", 1, peso_kg)
# ... consulta ...